# Run the architecture-conformance experiments on a free GPU (Kaggle or Colab)

This notebook runs the **same code** as the project on a free cloud GPU with Ollama, then packages the results for download.

**Current goal: the HARD TIER run (section 6).** Sections 1-5 are one-time setup; run them in order every new session.
The "Later" section at the end holds the confirmatory runs (primary, minimal prompt, held-out wording). Do not run those yet.

**Before you start**
1. **Kaggle:** create a notebook, set *Accelerator* to a GPU, turn *Internet* on. Add the project zip (any file named `architecture-analyzer*.zip`; use the Kaggle zip, which has no `frontend/` folder because Kaggle rejects `__name__` folders) as a dataset or upload it.
   **Colab:** *Runtime > Change runtime type > GPU*, then upload the zip in the Files panel.
2. Sessions end; the **cache** (saved in the results folder) lets you resume by re-running the same cell.


In [ ]:
# ---- settings you may change -------------------------------------------------
import glob, os
ZIP_PATH   = (glob.glob('/kaggle/input/**/architecture-analyzer*.zip', recursive=True)
              + glob.glob('/content/architecture-analyzer*.zip')
              + glob.glob('./architecture-analyzer*.zip'))
ZIP_PATH   = ZIP_PATH[0] if ZIP_PATH else None
# Kaggle automatically UNZIPS uploaded zips, so the project is usually already a folder under /kaggle/input.
SRC_DIR    = None
if not ZIP_PATH:
    hits = glob.glob('/kaggle/input/**/backend/experiments/run_experiments.py', recursive=True)
    if hits:
        SRC_DIR = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))   # .../<project>/backend/experiments/x.py -> .../<project>
BASE_MODEL = 'qwen2.5:7b-instruct'   # check ollama.com/library for a current 7-8B *instruct* model; name may differ
NUM_CTX    = 4096                    # must exceed the longest prompt (~1,100 tokens) plus the reply
WORK       = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
print('zip:', ZIP_PATH, '| project folder:', SRC_DIR, '| work dir:', WORK)
assert ZIP_PATH or SRC_DIR, 'Project not found: add the Kaggle zip as a dataset (Input panel) and run this cell again. Listing /kaggle/input:\n' + '\n'.join(glob.glob('/kaggle/input/*/*')[:20])


## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'NO GPU DETECTED - turn on a GPU accelerator, or expect very slow runs'

## 2. Unpack the project and install its Python dependencies

In [ ]:
import subprocess, os, shutil
os.chdir(WORK)
shutil.rmtree('architecture-analyzer', ignore_errors=True)
if ZIP_PATH:
    subprocess.run(['unzip', '-q', '-o', ZIP_PATH, '-d', '.'], check=True)
else:
    shutil.copytree(SRC_DIR, 'architecture-analyzer', copy_function=shutil.copyfile)   # /kaggle/input is read-only; work on a copy
os.chdir(f'{WORK}/architecture-analyzer/backend')
!chmod -R u+w {WORK}/architecture-analyzer
!pip install -q -r requirements-dev.txt
!python -m pytest tests -q --ignore=tests/test_frontend_contract.py 2>&1 | tail -3
print('Expect: 452 passed, 5 skipped (frontend test excluded). If you see 429 or ~378 you are running an OLD zip.')


## 3. Install and start Ollama
If the install line fails, read its error: newer installers need `zstd` (`apt-get install -y zstd`), and some
platforms need `sudo`. The server must answer on port 11434 before continuing.

In [ ]:
!apt-get install -y -q zstd > /dev/null 2>&1 || true
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request, os
env = dict(os.environ, OLLAMA_NUM_PARALLEL='1', OLLAMA_CONTEXT_LENGTH=str(NUM_CTX))
server = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2); print('Ollama is up'); break
    except Exception:
        time.sleep(2)
else:
    print(open('/tmp/ollama.log').read()[-2000:]); raise SystemExit('Ollama did not start')

## 4. Pull the model and give it a bigger context window
Ollama silently truncates prompts longer than its context window. The Modelfile below sets `num_ctx`, and temperature 0
keeps answers repeatable.

In [ ]:
!ollama pull {BASE_MODEL}
open('/tmp/Modelfile', 'w').write(f'FROM {BASE_MODEL}\nPARAMETER num_ctx {NUM_CTX}\nPARAMETER temperature 0\n')
!ollama create arch-judge -f /tmp/Modelfile
!ollama show arch-judge | tee {WORK}/model_info.txt

## 5. Smoke test the model through the project's own client (checks JSON output and speed)

In [ ]:
import os, sys, json
os.environ.update(LLM_BASE_URL='http://localhost:11434/v1', LLM_MODEL='arch-judge', LLM_API_KEY='ollama')
sys.path.insert(0, f'{WORK}/architecture-analyzer/backend')
from analysis.semantic.llm_client import client_from_env
from analysis.semantic.llm_analyzer import parse_verdict_json, REVIEW_VERDICTS
c = client_from_env()
r = c.complete('You are a reviewer. Reply with ONLY JSON.',
               'Reply with {"verdict":"violation","confidence":0.9,"explanation":"test","evidence_used":[]}')
print('ok:', r.ok, '| latency %.1fs' % r.latency_s, '| tokens in/out:', r.input_tokens, r.output_tokens)
print('raw:', r.text[:200]); print('parsed:', parse_verdict_json(r.text, REVIEW_VERDICTS))

## 6. HARD TIER (current goal)
Harder-to-read ADRs plus two mixed-decision scenarios; same labels. See `docs/HARD_TIER.md`.

### 6a. Generate it and count the calls (no model calls yet)
Expect `Generated 72 cases`, M3 = 72 calls, M5 = 57.

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard --seed 7 --wording hard
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --dry-run --seeds 5 --methods deterministic_adr,full_hybrid,evidence_guided,generic_selective,no_history,no_intent

### 6b. Core run (about 125 new calls, roughly 5-15 minutes)
`--max-calls` stops safely if the session is about to hit a limit; **re-run the same cell to resume from the cache.**

In [ ]:
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --splits dev,test --methods static,deterministic_adr,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/hard --max-calls 400 --seeds 5

### 6c. Sanity check: did the model's answers parse?
Good: `parsed ok` almost equal to `calls`, errors 0. If many fail to parse, stop and report it.

In [ ]:
import pandas as pd
p = pd.read_csv('../results/hard/predictions.csv')
p = p[p.llm_called.astype(bool)]
print('calls:', len(p), '| parsed ok:', int(p.parsed_ok.sum()), '| errors:', int(p.error.notna().sum()),
      '| median latency: %.1fs' % p.latency_s.median())
print(open('../results/hard/summary.md').read()[:3500])

### 6d. OPTIONAL: M2, the LLM-only baseline (about 500 calls, 30-45 minutes). Skip if GPU quota is short.

In [ ]:
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --splits dev,test --methods llm_only --out ../results/hard_m2 --max-calls 600

## 7. M7 (extract-then-compute): dev check
M7 keeps M5's escalation but has the model *extract* facts from the ADR (without showing it any date) and lets code compare dates. Iterate on **dev only**. Re-running reuses cached answers, so only the new extraction calls cost anything (about 25-30 calls).

**Look for:** scenario D accuracy (M5 scored 0 of 6), `parsed ok` close to the number of calls, and few `fallback` rows.

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --splits dev --methods static,evidence_guided,extract_compute --out ../results/hard_m7_dev --max-calls 300 --bootstrap 200

In [ ]:
import pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120)
p = pd.read_csv('../results/hard_m7_dev/predictions.csv')
t = p[p.truth != 'conformant'].copy(); t['correct'] = t.predicted_violation == (t.truth == 'violation')
print(t[t.method.isin(['M5_evidence_guided','M7_extract_compute'])].groupby(['scenario','method']).correct.mean().unstack().round(2))
m7 = p[(p.method == 'M7_extract_compute') & p.llm_called.astype(bool)]
print('M7 calls:', len(m7), '| parsed ok:', int(m7.parsed_ok.sum()), '| fallbacks:', int((m7.outcome_source == 'fallback').sum()))
print(m7[m7.scenario.str.startswith('D')][['case_id','truth','verdict','computed']].to_string(index=False))

### 7b. Prompt x2 (only if the dev check showed the J regression)
Run this patch cell, then re-run cell 7 (the dev run) and the check cell below it. Only M7's ~30 extraction calls are repeated; everything else comes from the cache.

In [ ]:
# Patch for a session that still has prompt x1 (no restart or re-upload needed). Safe to run twice.
path = f'{WORK}/architecture-analyzer/backend/analysis/semantic/extract.py'
s = open(path).read()
if 'EXTRACT_VERSION = "x2"' in s:
    print('already x2')
else:
    old_yes = '    "yes"     the source module is named, or described unambiguously as the same code;\n'
    new_yes = ('    "yes"     the source module is named, or described unambiguously as the same code; or the introducing commit cites this ADR '
               '(see the history notes and the "Selected because" line). A decision that describes the code by its purpose rather than by file name '
               'still covers it when the commit cites it;\n')
    old_no = '    "no"      the decision is about a different module, or this module is expressly excluded;\n'
    new_no = '    "no"      the decision names or describes a different module, or expressly excludes this one;\n'
    assert old_yes in s and old_no in s and 'EXTRACT_VERSION = "x1"' in s, 'extract.py is not the expected x1 file'
    s = s.replace(old_yes, new_yes).replace(old_no, new_no).replace('EXTRACT_VERSION = "x1"', 'EXTRACT_VERSION = "x2"')
    open(path, 'w').write(s)
    print('patched to x2')
!grep -n 'EXTRACT_VERSION =' {path}


### 7c. Diagnostic: what did M7 extract for the mixed-decision cases (L, M)?

In [ ]:
j = p[(p.scenario.str[0].isin(['L','M'])) & (p.truth != 'conformant') & (p.method == 'M7_extract_compute')]
print(j[['case_id','truth','verdict','computed','extraction']].to_string(index=False))

### 7d. DEVELOPMENT seeds 21 and 22 (144 cases; prompt iteration happens here, NOT on the confirmation seeds 11-13)
About 200 new calls (roughly 10 minutes). `--max-calls` stops safely; re-run the cell to resume.

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s21 --seed 21 --wording hard
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s22 --seed 22 --wording hard
DEV = 'static,evidence_guided,extract_compute'
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s21 --splits dev,test --methods {DEV} --out ../results/dev_s21 --max-calls 300 --bootstrap 100
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s22 --splits dev,test --methods {DEV} --out ../results/dev_s22 --max-calls 300 --bootstrap 100

### 7e. Pool the development seeds and read them

In [ ]:
!python -m experiments.combine_runs --runs ../results/dev_s21 ../results/dev_s22 --out ../results/dev_all --bootstrap 500
import pandas as pd
pd.set_option('display.width', 220)
q = pd.read_csv('../results/dev_all/predictions.csv')
t = q[q.truth != 'conformant'].copy(); t['correct'] = t.predicted_violation == (t.truth == 'violation')
print(t[t.method.isin(['M5_evidence_guided','M7_extract_compute'])].groupby(['scenario','method']).correct.agg(['mean','sum','count']).unstack().round(2))
m7 = q[(q.method == 'M7_extract_compute') & q.llm_called.astype(bool)]
print('M7 calls:', len(m7), '| parsed ok:', int(m7.parsed_ok.sum()), '| fallbacks:', int((m7.outcome_source == 'fallback').sum()))

### 7f. x4: the x2 prompt plus code-side grounding of the extracted limit date (x3 was tried and rejected)
Patch, run on the SAME development seeds (answers come from the cache), compare. The adoption rule is printed as PASS/FAIL lines.

In [ ]:
# Install extraction x4 = the x2 prompt (unchanged, so cached answers are reused) + code-side grounding of the limit date.
# Safe to run twice; the checksum catches a damaged copy-paste.
import base64, zlib, hashlib
BLOB = """
eNqtW21z20hy/s5fMcGmSoCXYuzNZlNFn9anWNpdpSw7Zeku2ZIVGASGElYgwMIAkngS/3ue7p4BhiBp+yrHDzI5L909/d494yAITh+bOkmbw+
ZWl4dptVi2jVa1NlXRNnlVqmquEjUvkpsbnalML3WZ6TJdqXChm9sqU+f/Hk1Go8tbrRY6MW2NVcu6mhV6ocLbpM5Uk+t6rLIqNf/y2/HHk/jy
7PTjZJGppMQctv3w8oefDl+9Ipxt0Zhoqh5ACqbV8clHdVNj3TzBwtqMOvS5Niovm7rK2hQI26VqKpCZJY0eK32v65V69+5cWRJNWhFZL+ksPy
mcidAm80bXh2nbVPO5ShOjJ4oPUWW6GNU6EeKIBOyuddoUK0dyCdz3umy0rCGsijiX1LkB9DDwSJvpOZDzunmt9d9keeB4Ns8fVW6I9Pw+N5bf
RTKrWvCsrBrMzHQDSompi2UzHY0UPnQ4pYhKw6BBX1ULS0//5/Lj8dvLCzWHVMGlhuDfVGBD0kwhBi07Mp0KvqWuF1hUMYbbfJY3Y0YhH16P6b
QCU9Xlb2cXxKG2AJsJWS7AGIZhcEUOaMSYSu2T1kPe3OYQsIemyRca42VWPaiQIDKYA6PuMMaY0qQAsKS2Mk6rZQ5QczCFKWj0YwOeEqy3H05O
6e/5f/3l8vSCZ0F7lqc4Pp+EBGW54KhiTWc5hgzy17yJmE5epcVEgI/JsqITgdCCWmQMxclnugYUqAptvq2KbA+eMQ+bMlmaW0iZUTuD6I3swI
ySG6w1FU+wbqrbxJBmgIU3zGUofg1k0PU8ZbQTdUoWYFd0+0jNzJ3ORN9Id3geYzjQr3XVljwQJllGi0r1+ONYrAQU3euiWi6g8wr0PljFF7gQ
xcphgrlAaNVD6TSSD4bTywGYfdB5XRCC0U9ikPMkLwjAIn/U2WGnmGST5A3gCAYCsFBxjDTVSxquSrA8nwNJ0s8+gC0NGyshH3WASVegCUYX84
k6luW8MfeNyZ7DAav1H5opAPcDMUymJXht/Y5KZob406mkwzcSfARiWSTsE1YifRy60A6/Z5JsdGlVNklejskyCz1vVFuyV74n4cxF7SB5gCF/
lxS0rqxKPWHlTG91ekdIU8iIFQhsK7XOSHec6qa3SXnjLLkRtHS0pmrTW3jhpNmhPlCWXyAwOHpoYpmx5DCnH5dFngIKG2tVGl3fJ01+r1mAbQ
mbMzpBWGA2rMYymhYaJg3/vCQylGmSXsikyiMQ2xreJlaDQ/LGu5KEs22+SVGAW8m9cNQ0NSZb4o51ASq8z6sioQ0RKQZLhGgeESvheHVRgILP
cwCaJendZ3hh4q+6abUxdPogCEYs4ziet4Ct41jli2VV09HBPYZtRiM79gcigvtea9kJSpO0SADPuK3dULdCs0/sp+3eZrVkjsv4cQlOflgSyq
QYyYoEX1cmNxN9n5MX0RPE7lrfJA2YZ/ed2qn/gNEXerDP6EVSwplMimIRp0VOWm33Ie685YHBFggCgxNyKrpD4vQ5NpolNIbll1lMZzH7UIpm
dmyJa/KEdZxk9WhkI1v819OPF2cf3qsjFTz+GCj1HTwVfpDAH3+wuj0m5WL1zsYQciuGcGhwbKQUztdV813+fTQ6OX17RiguAPYpkAhpgrEKXI
jkH2QvMSQO6jKYpgnWo7cfiDjetdK8qKzor9V0rHh3dn52KSvIWmmyD40xmFXVseQMg6mdg+yeAXV08fvF5el5bFlErAmC36uW3Ty8RHqLoEyW
4HkZcXNGhXDaJmK7hWkS1yVzcI5sMVEECXHm/YdL3g4eIkUjDzSIVxJaOgsTz7LC5kVrGt5uoy8rtCFrpvBVrqzxdnFsqioGb6p588DrJW9JKP
geI3dA2rZgX8GODitdhgqe+gTBW8/AgzuSdkLEFwmlgh5H4Btq5DMT9dHL+UIwBLImK8k8ziD7OoTcLQeDae8jJfB1vDXJCpxAFkezuU1jKJNG
wDJG8pROr7w0yI8C8EDVgxnbIMfMaqqC8wtOycIcStWyHuOU8IpdNIQ2Q9cpr8Ue/Uij5O9eW7SdDu9ECy2b5RlhGNugJ99JVmBfXpIgILQl8Q
cOqrFgt6yhy59zxwpTUUJOBOsC+Tb2kSgLqHDJoUKi8EPiRdu8ZJNNevp4jWWdJPLM0z56LZK7QTxNutQWvwjt2U1JCRspPkRYGc2RB04iFxDQ
fyInNC0QweuTE3EkjUm9YDukfcg4oQ+cG5uYBB33qhfsSrbBNSQqYB9n0gZZfqptQv1G2EhuoxPKxgImLlkQBSwLk9ZIN1H8YHCW37RVawpWeN
6JhezyXrPeeCkosR9WSBEW4ZoJBGCSU2i0MA4DiBUrYoU2XVoaXOhChDLD6VujA0XaEHEOtSEdR5uQQlRQykPyWrb1kthdJ9Z7QG0xNc9xPDob
5JAXhVQbrORcCgoUj+RO6eBdd6gwQTI+k6TAms+RMiGWuQKGjWNJugq+wUxgTI4d8CIWg/Pc7MZSjvBIIIuCJS8Z4HRHGQSoTZ2n31gJ+fXPG3
cyBAe164OUiL0lI7dU7gshylpKlyNjVASyXT1TEth7FMaxDb2DK59vgC58wKSthPdBllCm/g7IUpsIvF4cnF9AJqw0ftEo0R7qKellMiwQ1EaB
QCUmivzaSvB3fA7Pzw9PTlhvyhYaoN6i8uk6AK9togicaVswPiTj6mzuVT7OvSizBK456rVNCsM562SyWMIeoHowBsnA85uaIyrHz0LDBoIIKY
5xgK26kAyZtNHoozbLqhQVUx/ev/udI+N/XiBvqmbkyai9YKMsnXc6egqQZcV5Bt4Ff8ozlxyRZ1hxJlHkLI2OAT9zEtQHwz6gPW9EmecdwWH8
BbcpPvBZrPu5N8Fxb3D2vM9f0PznnXr7vEPnxgPNCXph0wY6KpZQgZOULAXmUEKOBFlN81ApKvwonTY/B2uqD0YoNufK5q7Wm8bIOSlmhPrett
YQIKgieVbvcZhIHf5MtjIVwwiC36wTdlm8+u+zy98+/OWyS5eQGdxokoXrIky9ao0TLtK1mZRptmIv28WMGmmEg7xr1RbkzqEBi6XVdJuhTegc
vGzOcDqqpc70BiY3ugmD5B5FIRVqQTTtGju1RoJVquB9RfUXqckhlbV1VXQxhpIDt5WtnP1vrw8TSwXtQlo7wOoFtVgiRBD5VPP8Nj2XW1krLa
ZQwLVOaaMNccTRCXH/dnp8Yskx+c02MQg6BiKJMYm6hnT8aS3USNZ0pK7mwdl2GLb7pupT8MQEXx3YoYPrCbnPZRitPwXBtTvZHiZAocnaY4qm
cYqsl3XVEweTMUEOgoOHzASbYHD8pbyLd0n3x8simErpkPTedzPzn/R8H1BX637Tl4nxJCI54KKiZgd5bJ1AnatSSnVF3rVWPmCHnqoAZnTHKw
hD6CAVr+oEhggzvmnBXZ8a3thR0609dGt7pejFFajvFTmPyR9VXoZAdLULyXUUbdFCjrbWnI6k5Mr30cEJasr9VNszEhp24h4A9fAy0H2cvxBt
tSBfO5C8J6Ihx1tnPJ9Ku4ThRJvOzpbSsVTh4Yz7C9NBv2HT18HbNCyzjsDgu+/UL1s3DkHfl54HF73iTtWToJmINsdLJJdrFZIjg2OFQTer5f
YqGlxHG0AvkxriGQBtePBrQO2qXUDrHAehopPKzG4D/UCwXaurjRFEWTbb9fV0c6mkslzIrT34wVtKihLuaEiaz42TqbfEQqHsKTZlDrFztz8I
ORVxvtcnOvC/QxbHX+wfIKE3+Qw5GmjX9wmxphp6cQvw2hP495C4194JFxP8HasFNZFMVUL3KBosyPDsCTBvrkmfvd9cSaqrgEIoNmBiEgyxBG
QoOMZ7Lme4rbEvGoU2uTbcWb1BEVjSfZEtW3weuYCyM8RbAjtPGA33MUkfpdvCV0PccbG1m5ekcfLrTrRlgHxEMsDv1OH/+0MMo460mDOStXoR
3idFCyU/LlebJmspQZZg2llYB1efzOH193SmGH+wTHZGLnpNiupB12HknEWcm4qzrSEG18q8osnrqXNgvIrCNIl5K5zTII8t4EWIJk3tHZAVfs
qeflxHh/TvD/2/GyRupAuL/bCberU1ySdAEAoXV6+ukY/L1x/6r//qnLA0YdRfCeVpXVf1bkTCHW6VO09K/VPqIYecpBPhzCgvc+wyxl/yGlHS
1x4bxrnjLpUADo3UXiMP1FJRmNcMRAptyqqpyKWUkMz7nvzWMBOUamEfn7A7y8WEYHq01imBKPGcwkA2ENTnz5/DN1M6ZvTmk3kRfnqavHjzaR
3hO6YgL6nIsOXkw+Xxu3d9ZGNoPTU99klOVxBN+HJsF02o7bsMX1mJwDc1t6QIsCFQg2XdEcjv5Kj/bol/jlWaBeBFamDHiqMjFTwF0w3zxgxD
p8mX09Gwfnco840Z2QF39aobRsHV4VhLccerpjs2Hvobt4jgvYLX5rqbhrSDfTY9oENf8c6pypENQM+3dnGDteNcKo0ED1aPZ8OI6AM9BSdI8H
ARSWZC2tVj+ILZMLnw3nnZal8mnPXDj4FmSD0EgjEbS/SVrS6sjW3zaWwr6yPrCgFIcreu4IW3CyKY+mB+T1m7Z7UUtGNb0EYcmOXraEOSrrPF
NY3q7ye4LcLNMjtlbyDo0pDpt8Ny7fAVHkj9jyN35NmOQDQ096cNQH3jgF0JCLaQgDikobEEPKqkcUYO2eSBZdUgOPodhV4o+7sFTlxdb4D/Hc
DcqO/76LMpBpmPhvRsFv5Ed7fNnxIBO2fXw1iPtr38PyRe99dY4ZISM0rwEGDj89PL4/jk+PJUnCxlrEho4WX/l1zrm+nV4YtrfIOnfUE/6dDP
tu57bpecwDwXqLeuVHy9qLJ8nusswlrsuZoeXU9e/HPArvjMRXM+MuVwcrU3iFR+T+Okrpb0KCeh607XxaTSmF5dUGc2oacuY3VwgqnpgUxG3N
dY0Z1K3w62zwse+i6ePJfhC5nc9vhMF7p21i7srvhLLj2/iVkWOcSKVZHzJT07J6AO0Yp73u7o3RWnnBw8kDPrxs9gQMVbv8+31Xkc3B7xg4Fw
g0/GNaizqDtUd60ac+ILcQ9vXImgCcctnwtP84kkVHT+OdHQ38uGQ2FaSFG0tkcWtYuZweGj9LDGZPCGTM80XXrSnf3cXcjsfMaRlOaBepCVf+
PB0lAfmV5q3tMbE0lgPndq/5nkw73qcmWfMkzkdVR3EdbdDOx8EbLJcMshe+803Ln/cYt9BsXKtxv8tHtdkqhbuSHn/t1oR2u/e1fSPfoRVs2A
kS41Dig4HFha7XOSIa3W5W+dEHVTszlq35iQciXco+fbU7I97i0z2S4r8p6fKOkbP+RY4yombkTalyh96I+sO44dbdzJpVsBORwpHomQnlsYRQ
8unNLwv3RpeMTKFD5GbuQq6FQguKabblAgTEqLJF/oLCDbfbyyAeGa8yfpF3PkCZx6dFntrrVbGS4wS4JL94UlJbdcsfZVKpejAs3Gw2tWDl42
obh65M9dE8f2wbB7TCw3kbabBTfjJHdEPgaFVVuSmb8Ih45IqueEEzEmOIokY2XiJQITAKtJpM6xy9qfkgmLawCB4zrPSEgjh+M4J6Gza7plfp
opiSvB8Q6wl737d/tU9tuxbyJRK2RKjtyzip6uI0lBOq05CpyZT7/Nxse+Ye6zcpskcdIOXx96h41I9K96mkOq6scR1QD9or0nkjPwFu8M88DZ
/1Q9ZeuvuKDXfPvKN2nutE/cWkAqBDEjsoXRujuB0dOvUDPk6Dzo3cO30DNGoM5R2tBbxqcht9a9A5HYt9mAJE35B6VQtqO6bOXV1mYi9ef+TR
b/RThyj6KnzheRU+ZUh14hHfzRmobTpQOU5gfdMxj+1fNHvDfdrfNOCVl/XiI1Qr26ssXpnG4jMr5AjjtAIT1Z5Pg6q6piy4T4QaMlSv0THFlH
Dz+S8hz4ndZLs+eFXNfP5bVeG6IP9n3ffTro2DBxQz7Bm18SfX34l3YXP8siIn7Nmx1P+fjVaOLImsAbuJeI5T0fw29MwBF1hnjNh3eXkVts6s
kLg461cBjzDoJ6erw6cD8Ortf93QbQ7ClEXOj4dnyOwjHHZTKW7vpLXih8O1p3Q/pF3L38t5DzA0z3HHMD7dfjoo+iV7geg3+i8ebjhcDdjjm/
MlSn/RFmM6vY2Rbce/a5QGQpyzvz67V7eko5JmUdif/stOeId//1d6O1WC0il4vt0Hvj3rdavNUdPxfcd9M99an605HHGj/D3HEZvrnxG/bJdf
nGtp+9beurXmGu3TUppSpPlzU1eQMD1TdQD7oa/SWhMCMvA1j5ZXIVrK+qu41+924FI2mAMZLReWY12vlmBjL3iH7yLg79AKieiOC1OtjQjKf+
hBurD4LoHxqHkAKPbKR59+7c/recY3oh+zfdl9AfSllKD2SUNvjK6mQT6bFc2c7ajC6yEvee/T7XD/LYFqBUeP5vUkR2sSaO8xJHjDnAjJW8+Z
32r33HqMwf46a606Vh8UOoP7586fXUOPakdjH/6HdgcboFpkduw4LF/YUbw76oFO0wS06DO8QT9zgm3HwNO/7i5WQ05tcP1ONF8D96OXnpk3k0
OEvfCeMuPeXAu9v1RJ6U3qSp/Mupa9dotk7FQtrbme0wbZTeMjr2L8O+1qaj8ElvVyytwbj7GtsLMcwOXlmTI2dMmHIo7Uhc3XWDPvXD/hk1br
GQecA/XG9lgyuh6x1agGLZ/n8ckOgP74x8LRh26XjSYeEfoFNkFrdCPrlQ2moX8Q8rVqy1cO2k/LKz6wEuC1bDF/EjZLdpOE4+P6GHQavYuDXd
APU0k/S23y2/Bqjq5CEmLepQ0EWI11v8P8PgLdU=
"""
raw = zlib.decompress(base64.b64decode(''.join(BLOB.split())))
assert hashlib.sha256(raw).hexdigest() == "d15f154c1fcc6f94a70d50c914f1cccc0994d492b91bb4efebbc57eae094d946", 'checksum mismatch: the text was damaged when copied; copy the whole cell again'
path = f'{WORK}/architecture-analyzer/backend/analysis/semantic/extract.py'
open(path, 'wb').write(raw)
print('installed', len(raw), 'bytes')
!grep -n 'EXTRACT_VERSION =' {path}


In [ ]:
%cd {WORK}/architecture-analyzer/backend
# Regenerate the development seeds only if this is a fresh session (they are deterministic, so they come out identical).
!test -d ../datasets/benchmark/generated_hard_s21 || python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s21 --seed 21 --wording hard
!test -d ../datasets/benchmark/generated_hard_s22 || python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s22 --seed 22 --wording hard
M7 = 'static,evidence_guided,extract_compute'
# x4 uses the x2 prompt. In a session that still has the cache, answers come from it (few or no new calls).
# In a fresh session there is no cache, so expect about 200 real calls (roughly 10 minutes); --max-calls stops safely, re-run to resume.
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s21 --splits dev,test --methods {M7} --out ../results/dev_s21_x4 --max-calls 300 --bootstrap 100
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s22 --splits dev,test --methods {M7} --out ../results/dev_s22_x4 --max-calls 300 --bootstrap 100


In [ ]:
import os, json
import pandas as pd
from experiments import metrics
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 200)
def load(tag, strip=''):
    d = pd.read_csv(f'../results/{tag}/predictions.csv'); d['case_id'] = tag.replace(strip, '') + ':' + d['case_id'].astype(str); return d
def both(a, b, strip=''):
    tags = [a, b]
    return pd.concat([load(t, strip) for t in tags]) if all(os.path.exists(f'../results/{t}/predictions.csv') for t in tags) else None
def pick(d, name):
    return d[d.method == ('M5_evidence_guided' if name == 'M5' else 'M7_extract_compute')].assign(method=name)
x4 = both('dev_s21_x4', 'dev_s22_x4', '_x4'); x2 = both('dev_s21', 'dev_s22'); x3 = both('dev_s21_x3', 'dev_s22_x3', '_x3')
frames = [pick(x4, 'M5'), pick(x4, 'M7_x4')] + ([pick(x2, 'M7_x2')] if x2 is not None else []) + ([pick(x3, 'M7_x3')] if x3 is not None else [])
print('columns available:', [f.method.iloc[0] for f in frames], '(x2/x3 are missing if this is a fresh session: that is fine)\n')
allp = pd.concat(frames); allp['split'] = 'dev'
print(metrics.summarize(allp, n_boot=300)[['method', 'precision', 'recall', 'f1', 'f1_ci_low', 'f1_ci_high', 'llm_calls']].round(3).to_string(index=False))
t = allp[allp.truth != 'conformant'].copy(); t['correct'] = t.predicted_violation == (t.truth == 'violation')
tab = t.groupby(['scenario', 'method']).correct.sum().unstack().astype(int)
print(tab)
for b in [m for m in ('M5', 'M7_x2') if m in set(allp.method)]:
    print(metrics.paired_bootstrap(allp, 'dev', ['M7_x4'], [b], 500).query("metric == 'f1'").round(3).to_string(index=False))
m = allp[(allp.method == 'M7_x4') & allp.llm_called.astype(bool)]
fb = int((m.outcome_source == 'fallback').sum()); f1 = metrics.summarize(allp[allp.method == 'M7_x4'], n_boot=10).f1.iloc[0]
g = lambda L: int(tab.loc[[s for s in tab.index if s.startswith(L)][0], 'M7_x4'])
print('\nM7 x4 calls:', len(m), '| parsed ok:', int(m.parsed_ok.sum()), '| fallbacks:', fb)
print('\nADOPTION RULE for x4 (fixed before this run, docs/ANALYSIS_PLAN.md):')
for label, ok in [(f'F1 >= 0.964   (got {f1:.3f})', f1 >= 0.964), (f'D >= 11/12     (got {g("D")})', g('D') >= 11), (f'J >= 11/12     (got {g("J")})', g('J') >= 11),
                  (f'M >= 10/12     (got {g("M")})', g('M') >= 10), (f'L >= 8/12      (got {g("L")})', g('L') >= 8), (f'fallbacks <= 10% (got {fb}/{len(m)})', fb <= 0.1 * len(m))]:
    print(('PASS  ' if ok else 'FAIL  ') + label)
print('\nGrounding actions taken by x4 (rows with an extraction; history-only escalations use the review prompt):')
print(m.extraction.dropna().map(lambda s: json.loads(s).get('grounding', '?').split(':')[0]).value_counts().to_string())


## 8. CONFIRMATION of the frozen M7 (prompt x4) on fresh seeds 11, 12, 13
Run ONLY after the x4 patch cell. The first cell refuses to continue unless the code matches the freeze record. About 350 real calls. Criteria C1-C5 are in `docs/ANALYSIS_PLAN.md`.

In [ ]:
# Verify the code is EXACTLY the frozen M7 x4 (refuses to continue otherwise), then generate the three confirmation seeds.
import hashlib, os
FROZEN = {
 "analysis/semantic/extract.py": "d15f154c1fcc6f94a70d50c914f1cccc0994d492b91bb4efebbc57eae094d946",
 "analysis/semantic/prompts.py": "41a0487715fd125fc93e485dd9cfc8e935c2c33c629abe557f0ae7a7bdf0e084",
 "analysis/semantic/llm_client.py": "9c83d782b924a4cc5dd8e75640eabc2059c8802bc43c861944f6bf9bf0543f47",
 "analysis/policy/escalation.py": "8c4b4642e0217482cb7534b24a21565dde1482b75c4a6f7cfc48ffdfd4a41454",
 "analysis/evidence/aggregator.py": "dcc219ca9a5f5fb5e7426a12872d00244062d50aebd3c7c654d25b1b8961395b",
 "analysis/intent/adr.py": "9730d50f9534bd02d009f902819e4e49069a6c01b9bf4a604e8ae95b20eda612",
 "analysis/intent/reader.py": "5b06575f23a0e891799f1570b57e0f309f403ba342e0f9e382cacc5a6ad900d1",
 "analysis/evolution/git_evolution.py": "d6749cfa0e76737c212a2b3c268bdc778453eb2778e1bc19d777bcd91c889894",
 "experiments/pipeline.py": "b1febe431bb35cc996c9f64c01e602dd8a905c4035adcae97e61cb21aaeaa97e",
 "experiments/metrics.py": "01296ce8df991c818cb588579c5c8705b9ad5a1117ecd7edfe5662d0021c1b70",
 "experiments/run_experiments.py": "4957361e713765e992813fe6c3580e12442004c084b52afc17b24ea6e49d8035",
 "experiments/combine_runs.py": "4221bdc0094e79a48eb862aa8c6561ed86e4e225515a4ff23701a72e68592345"
}
root = f'{WORK}/architecture-analyzer/backend'
bad = [f for f, h in FROZEN.items() if hashlib.sha256(open(f'{root}/{f}', 'rb').read()).hexdigest() != h]
assert not bad, 'NOT THE FROZEN CODE. These files differ: ' + ', '.join(bad) + ' (re-run the x4 patch cell, or tell me)'
print('frozen code verified:', len(FROZEN), 'files match the freeze record (docs/frozen_m7_x4_sha256.json)')
%cd {WORK}/architecture-analyzer/backend
!test -d ../datasets/benchmark/generated_hard_s11 || python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s11 --seed 11 --wording hard
!test -d ../datasets/benchmark/generated_hard_s12 || python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s12 --seed 12 --wording hard
!test -d ../datasets/benchmark/generated_hard_s13 || python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s13 --seed 13 --wording hard


In [ ]:
# Run the frozen methods ONCE per seed. About 350 real calls in total (roughly 20 minutes). --max-calls stops safely;
# if the session stops, run this same cell again and it resumes from the cache. Do NOT change anything between runs.
M = 'static,deterministic_adr,full_hybrid,evidence_guided,extract_compute'
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s11 --splits dev,test --methods {M} --out ../results/confirm_s11 --max-calls 450 --bootstrap 100
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s12 --splits dev,test --methods {M} --out ../results/confirm_s12 --max-calls 450 --bootstrap 100
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s13 --splits dev,test --methods {M} --out ../results/confirm_s13 --max-calls 450 --bootstrap 100


In [ ]:
# Pool the three seeds (the tool refuses to pool runs that differ in model, prompt version or method set).
!python -m experiments.combine_runs --runs ../results/confirm_s11 ../results/confirm_s12 ../results/confirm_s13 --out ../results/confirm_all --bootstrap 1000


In [ ]:
# The pre-registered verdict (criteria fixed in docs/ANALYSIS_PLAN.md before any M7 result existed). Report it whatever it says.
import pandas as pd
pd.set_option('display.width', 220)
q = pd.read_csv('../results/confirm_all/predictions.csv'); pc = pd.read_csv('../results/confirm_all/paired_comparisons.csv')
M7, M5 = 'M7_extract_compute', 'M5_evidence_guided'
t = q[q.truth != 'conformant'].copy(); t['correct'] = t.predicted_violation == (t.truth == 'violation')
isD = lambda d: d.scenario.str.startswith('D')
acc = lambda m, sel: t[(t.method == m) & sel(t)].correct.mean()
c1 = acc(M7, isD); n7, n5 = acc(M7, lambda d: ~isD(d)), acc(M5, lambda d: ~isD(d))
f1 = pc[(pc.a == M7) & (pc.b == M5) & (pc.metric == 'f1')].iloc[0]
calls = q[q.llm_called.astype(bool)].groupby('method').size()
esc = q[(q.method == M7) & q.escalated.astype(bool)]; fb = int((esc.outcome_source == 'fallback').sum())
print('cases pooled:', q.case_id.nunique(), '| M7 calls:', calls[M7], '| M5 calls:', calls[M5], '\n')
print('PRE-REGISTERED CRITERIA (M7 = frozen prompt x4)')
for label, ok in [(f'C1 scenario D accuracy >= 80%            (got {c1:.0%}; M5 got {acc(M5, isD):.0%})', c1 >= 0.80),
                  (f'C2 paired F1 M7 - M5, lower 95% bound > 0 (got {f1.delta:+.3f}, CI {f1.ci_low:+.3f} to {f1.ci_high:+.3f})', f1.ci_low > 0),
                  (f'C3 calls <= 105% of M5                    (got {calls[M7]} vs {calls[M5]})', calls[M7] <= 1.05 * calls[M5]),
                  (f'C4 non-D accuracy within 0.05 of M5       (got {n7:.3f} vs {n5:.3f})', n7 >= n5 - 0.05),
                  (f'C5 fallbacks <= 10% of escalations        (got {fb}/{len(esc)})', fb <= 0.10 * max(len(esc), 1))]:
    print(('PASS  ' if ok else 'FAIL  ') + label)
print('\nAccuracy on the targeted dependency by scenario (pooled):')
print(t[t.method.isin([M5, M7, 'M3_full_hybrid', 'M6_deterministic_adr'])].groupby(['scenario', 'method']).correct.mean().unstack().round(2))
print('\nAll paired F1 comparisons:'); print(pc[pc.metric == 'f1'][['a', 'b', 'a_value', 'b_value', 'delta', 'ci_low', 'ci_high']].round(3).to_string(index=False))


## 9. Package for download
Kaggle: *Output* tab. Colab: Files panel. Upload `results_m7.zip` back to the conversation. (The cache file is left out on purpose.)

In [ ]:
!cp {WORK}/model_info.txt ../results/hard/ 2>/dev/null || true
!cd .. && zip -qr {WORK}/results_m7.zip results -x '*.sqlite'
print('Download', f'{WORK}/results_m7.zip')

---
# LATER: confirmatory runs (do NOT run these yet)
These are the primary, minimal-prompt and held-out runs from `docs/ANALYSIS_PLAN.md`. Run them only after the policy and prompts are frozen and the plan is committed to Git.

## LATER 6. Generate the benchmark and count the calls (no model calls yet)

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated --seed 7
!python -m experiments.run_experiments --dry-run --seeds 5

## LATER 7. PILOT: one case per scenario (10 cases)
Look at the table, then at `predictions.csv`: is `parsed_ok` true? are `explanation`s sensible? is `latency_s` small?
If most calls fail to parse, fix that **before** spending the full run.

In [ ]:
!python -m experiments.run_experiments --limit-cases 10 --methods evidence_guided,full_hybrid --out ../results/pilot --bootstrap 100
import pandas as pd
p = pd.read_csv('../results/pilot/predictions.csv')
p = p[p.llm_called.astype(bool)]
print('calls:', len(p), '| parsed ok:', int(p.parsed_ok.sum()), '| errors:', int(p.error.notna().sum()),
      '| median latency: %.1fs' % p.latency_s.median())
p[['method','case_id','scenario','truth','verdict','predicted_violation','latency_s','explanation']]

## LATER 8. Dev split (iterate prompts HERE only, never on the test split)
Re-running after a prompt change re-calls only what changed (responses are cached).

In [ ]:
!python -m experiments.run_experiments --splits dev --methods static,full_hybrid,evidence_guided,no_history --out ../results/dev --bootstrap 200

## LATER 9. FULL run, primary setup (freeze prompts and policy first; `docs/ANALYSIS_PLAN.md` says what counts as support)
Without M2 first (99 distinct calls). `--max-calls` stops safely if the session is about to hit a limit; run the same cell again to resume from the cache.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --methods static,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/primary --max-calls 400 --seeds 5

### LATER 9b. M2 (LLM on every edge) — 420 calls, the expensive one. Run it ONCE, on the primary setup.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --methods llm_only --out ../results/primary_m2 --max-calls 600

## LATER 9c. Robustness runs (99 calls each; skip M2)
* **Minimal prompt**: same data, all scenario-specific guidance removed from the system prompts.
* **Held-out wording**: the same repositories with the ADR decision language rewritten. **Commit `docs/ANALYSIS_PLAN.md` to Git before this cell** so the plan provably came first.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --prompt-variant minimal --methods static,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/minimal --max-calls 400 --seeds 5

In [ ]:
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_heldout --seed 7 --wording heldout
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_heldout --splits dev,test --methods static,deterministic_adr,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/heldout --max-calls 400 --seeds 5

## LATER 10. Package everything for download

In [ ]:
!cp {WORK}/model_info.txt ../results/primary/ 2>/dev/null || true
!cd .. && zip -qr {WORK}/results.zip results
print('Download', f'{WORK}/results.zip', '(Kaggle: Output tab; Colab: Files panel).')
print(open('../results/primary/summary.md').read()[:3000])